# Recommenders Training (SPINRec-style Step 2)

统一训练三类基准模型：MLP、VAE、NCF。

In [1]:
%run ./functions.ipynb
%run ./baselines_functions.ipynb
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("nbformat") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nbformat"])

%run ./functions.ipynb

from pathlib import Path

def _find_root() -> Path:
    c = Path.cwd().resolve()
    for p in [c, *c.parents]:
        if (p / "DATASET").exists() and (p / "recacc").exists():
            return p
    return c

REPO_ROOT = _find_root()
seed_everything(42)

cache_path = REPO_ROOT / "recacc" / "log" / "notebook_cache" / "prepared_data.pt"
cache = torch.load(cache_path)

prepared = PreparedData(
    dense=cache["dense"],
    sparse=cache["sparse"],
    labels=cache["labels"],
    sample_ids=cache["sample_ids"],
    sparse_fields=cache["sparse_fields"],
    dense_fields=cache["dense_fields"],
)
NUM_BUCKETS = int(cache["num_buckets"])

train_loader, train_eval_loader, val_loader = make_loaders(
    prepared, batch_size=256, val_ratio=0.2, seed=42
)

print("Repo root:", REPO_ROOT)
print("Train batches:", len(train_loader), "Val batches:", len(val_loader))

C:\Users\vanvan\AppData\Local\Temp\ipykernel_33788\1394357569.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cache = torch.load(cache_path)


Repo root: H:\TZB
Train batches: 2431 Val batches: 608


In [2]:
# 覆盖版 train_model：兼容新版超参并启用提分训练策略（支持实时 checkpoint 落盘）
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 3,
    lr: float = 1e-3,
    device: str = DEVICE,
    verbose: bool = True,
    early_stop_patience: int = 0,
    early_stop_min_delta: float = 0.0,
    save_end_only: bool = False,
    checkpoint_interval_epochs: float = 0.5,
    use_class_balance: bool = True,
    weight_decay: float = 1e-5,
    max_grad_norm: float = 5.0,
    lr_scheduler_patience: int = 2,
    lr_scheduler_factor: float = 0.5,
    min_lr: float = 1e-6,
    checkpoint_dir=None,
    checkpoint_prefix: str = "model",
):
    print('new new')
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=float(weight_decay))
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="max",
        factor=float(lr_scheduler_factor),
        patience=int(lr_scheduler_patience),
        min_lr=float(min_lr),
        verbose=False,
    )

    def _format_epoch_tag(ep_value: float) -> str:
        ep_value = float(ep_value)
        ep_i = int(np.floor(ep_value + 1e-9))
        frac = ep_value - ep_i
        if abs(frac) < 1e-9:
            return f"epoch{ep_i:02d}"
        frac_str = f"{frac:.3f}".split(".")[1].rstrip("0")
        if frac_str == "":
            frac_str = "0"
        return f"epoch{ep_i:02d}_{frac_str}"

    checkpoint_paths = []
    checkpoint_records = []  # (epoch_value, path)
    ckpt_dir = Path(checkpoint_dir) if checkpoint_dir is not None else None
    if ckpt_dir is not None:
        ckpt_dir.mkdir(parents=True, exist_ok=True)

    def _save_ckpt(state_dict_obj, ep_value: float, is_final: bool = False):
        if ckpt_dir is None:
            return None
        if save_end_only and is_final:
            p = ckpt_dir / "final.pt"
        else:
            tag = _format_epoch_tag(ep_value)
            p = ckpt_dir / f"{tag}.pt"
        torch.save(state_dict_obj, p)
        p_str = str(p)
        checkpoint_paths.append(p_str)
        checkpoint_records.append((float(ep_value), p_str))
        return p_str

    pos_weight_t = None
    if use_class_balance:
        try:
            labels_all = train_loader.dataset.prepared.labels.detach().float().cpu()
            pos = float((labels_all > 0.5).sum().item())
            neg = float((labels_all <= 0.5).sum().item())
            # 仅在明显不平衡时启用，避免过度加权导致 AUC 不稳
            if pos > 0 and neg > 0:
                ratio = neg / pos
                if ratio >= 3.0:
                    pw = max(1.0, min(ratio, 20.0))
                    pos_weight_t = torch.tensor([pw], dtype=torch.float32, device=device)
        except Exception:
            pos_weight_t = None

    hist = []
    snapshots = []
    best_auc = -1e18
    best_state = None
    bad_epochs = 0

    ckpt_interval = float(checkpoint_interval_epochs)
    if ckpt_interval <= 0:
        ckpt_interval = 0.5
    next_ckpt_epoch = ckpt_interval
    stopped_epoch = int(epochs)

    for ep in range(1, epochs + 1):
        model.train()
        run_loss = 0.0
        n = 0
        for batch in train_loader:
            dense, sparse, y, _sid = _unpack_batch(batch)
            dense = dense.to(device)
            sparse = {k: v.to(device) for k, v in sparse.items()}
            y = y.to(device)
            logits = model(dense, sparse)
            if isinstance(logits, tuple):
                logits = logits[0]
            if pos_weight_t is not None:
                loss = F.binary_cross_entropy_with_logits(
                    logits.view(-1), y.float().view(-1), pos_weight=pos_weight_t
                )
            else:
                loss = _loss(logits, y)
            opt.zero_grad()
            loss.backward()
            if max_grad_norm and float(max_grad_norm) > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(max_grad_norm))
            opt.step()
            bs = int(y.shape[0])
            run_loss += float(loss.item()) * bs
            n += bs

        tr_loss = run_loss / max(n, 1)
        val_m = evaluate_model(model, val_loader, device=device)
        sched.step(float(val_m.get("auc", 0.0)))
        row = {
            "epoch": ep,
            "train_loss": tr_loss,
            "lr": float(opt.param_groups[0]["lr"]),
            **val_m,
        }
        hist.append(row)
        if verbose:
            print(row)

        if not save_end_only:
            if (ep + 1e-12) >= next_ckpt_epoch:
                st = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                snapshots.append(st)
                _save_ckpt(st, ep, is_final=False)
                while next_ckpt_epoch <= (ep + 1e-12):
                    next_ckpt_epoch += ckpt_interval

        if early_stop_patience and early_stop_patience > 0:
            cur_auc = float(val_m.get("auc", 0.0))
            if cur_auc > best_auc + float(early_stop_min_delta):
                best_auc = cur_auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= int(early_stop_patience):
                    stopped_epoch = ep
                    if verbose:
                        print({"early_stop": True, "stopped_epoch": ep, "best_auc": best_auc})
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    best_epoch_hist = int(pd.DataFrame(hist)["auc"].idxmax() + 1) if len(hist) > 0 else stopped_epoch

    final_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if save_end_only:
        snapshots = [final_state]
        _save_ckpt(final_state, stopped_epoch, is_final=True)
    else:
        snapshots.append(final_state)
        _save_ckpt(final_state, stopped_epoch, is_final=False)

    # 始终保存 best.pt，方便 benchmark 直接读取最优点
    if ckpt_dir is not None and best_state is not None:
        best_path = ckpt_dir / "best.pt"
        torch.save(best_state, best_path)

    # 删除最优 epoch 之后的所有 checkpoint
    if ckpt_dir is not None and len(checkpoint_records) > 0:
        kept = []
        for ep_value, p_str in checkpoint_records:
            if ep_value <= float(best_epoch_hist) + 1e-12:
                kept.append(p_str)
            else:
                p = Path(p_str)
                if p.exists():
                    p.unlink()
        if best_state is not None:
            best_str = str(ckpt_dir / "best.pt")
            if best_str not in kept:
                kept.append(best_str)
        checkpoint_paths = kept

    return pd.DataFrame(hist), snapshots, checkpoint_paths

In [3]:
# ===== 训练样本概览 =====
n_samples = int(prepared.labels.shape[0])
pos_rate = float(prepared.labels.float().mean().item()) if n_samples > 0 else 0.0
neg_rate = 1.0 - pos_rate

print("[Data Overview]")
print("samples:", n_samples)
print("positive rate:", round(pos_rate, 6))
print("negative rate:", round(neg_rate, 6))
print("majority-class baseline acc:", round(max(pos_rate, neg_rate), 6))
print("dense dim:", int(prepared.dense.shape[1]) if prepared.dense.ndim == 2 else 0)
print("sparse fields:", prepared.sparse_fields)

if "user_id" in prepared.sparse:
    n_users = int(torch.unique(prepared.sparse["user_id"]).numel())
    print("unique users:", n_users)
if "item_id" in prepared.sparse:
    n_items = int(torch.unique(prepared.sparse["item_id"]).numel())
    print("unique items:", n_items)

print("loader batches (train/val):", len(train_loader), len(val_loader))

[Data Overview]
samples: 777818
positive rate: 0.5
negative rate: 0.5
majority-class baseline acc: 0.5
dense dim: 1
sparse fields: ['user_id', 'item_id']
unique users: 15324
unique items: 9366
loader batches (train/val): 2431 608


In [4]:
OUT_DIR = REPO_ROOT / "recacc" / "log" / "notebook_runs"
STEP2_ROOT = OUT_DIR / "step2_training"
for p in [OUT_DIR, STEP2_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

from datetime import datetime

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
STEP2_DIR = STEP2_ROOT / RUN_TIMESTAMP
STEP2_DIR.mkdir(parents=True, exist_ok=True)

MODELS = ["MLP", "VAE", "NCF"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ===== 超参设置项（可直接修改） =====
TRAINING_DEFAULTS = {
    "epochs": 50,
    "lr": 2e-4,
    "batch_size": 256,
    "val_ratio": 0.2,
    "seed": 91,
    "early_stop_patience": 4,
    "early_stop_min_delta": 5e-5,
    "checkpoint_end_only": False,      # 只保存终点：True/False
    "checkpoint_interval_epochs": 0.5, # 保存频率（每个 checkpoint 间隔几个 epoch），仅在 end_only=False 时有效
    "use_class_balance": True,
    "weight_decay": 1e-5,
    "max_grad_norm": 5.0,
    "lr_scheduler_patience": 2,
    "lr_scheduler_factor": 0.5,
    "min_lr": 1e-6,
    "dropout": 0.1,
}

# 按模型覆盖默认超参；未指定项会回退到 TRAINING_DEFAULTS
# 这里已经应用 4 层宽度建议
MODEL_HPARAMS = {
    "MLP": {
        "lr": 2e-4,
        "epochs": 60,
        "emb_dim": 32,
        "hidden_dims": [512, 256, 128, 64],
    },
    "VAE": {
        "lr": 2e-4,
        "epochs": 60,
        "emb_dim": 32,
        "encoder_dims": [512, 256, 128, 64],
        "latent_dim": 64,
    },
    "NCF": {
        "lr": 2e-4,
        "epochs": 50,
        "emb_dim": 64,
        "mlp_hidden_dims": [256, 128, 64, 32],
    },
}


def resolve_hparams(model_name: str):
    cfg = dict(TRAINING_DEFAULTS)
    cfg.update(MODEL_HPARAMS.get(model_name, {}))
    return cfg


def _build_mlp_stack(in_dim: int, hidden_dims, dropout: float = 0.1):
    dims = [int(d) for d in hidden_dims]
    layers = []
    prev = int(in_dim)
    for d in dims:
        layers.append(nn.Linear(prev, d))
        layers.append(nn.ReLU())
        if float(dropout) > 0:
            layers.append(nn.Dropout(float(dropout)))
        prev = d
    return nn.Sequential(*layers), prev


class MLPRec4L(nn.Module):
    def __init__(self, sparse_fields, dense_dim, num_buckets, emb_dim=32, hidden_dims=None, dropout=0.1):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [512, 256, 128, 64]
        self.enc = SparseEncoder(sparse_fields, num_buckets, int(emb_dim))
        in_dim = int(dense_dim) + int(emb_dim) * len(sparse_fields)
        self.backbone, out_dim = _build_mlp_stack(in_dim, hidden_dims, dropout=dropout)
        self.out = nn.Linear(out_dim, 1)

    def forward(self, dense, sparse):
        x = torch.cat([dense, self.enc(sparse)], dim=1)
        h = self.backbone(x)
        return self.out(h).squeeze(1)


class NCFRec4L(nn.Module):
    def __init__(self, num_buckets, emb_dim=64, mlp_hidden_dims=None, dropout=0.1):
        super().__init__()
        if mlp_hidden_dims is None:
            mlp_hidden_dims = [256, 128, 64, 32]
        e = int(emb_dim)
        self.ug = nn.Embedding(num_buckets, e)
        self.ig = nn.Embedding(num_buckets, e)
        self.um = nn.Embedding(num_buckets, e)
        self.im = nn.Embedding(num_buckets, e)
        self.mlp, mlp_out_dim = _build_mlp_stack(e * 2, mlp_hidden_dims, dropout=dropout)
        self.out = nn.Linear(e + mlp_out_dim, 1)

    def forward(self, dense, sparse):
        u = sparse["user_id"]
        i = sparse["item_id"]
        gmf = self.ug(u) * self.ig(i)
        m = self.mlp(torch.cat([self.um(u), self.im(i)], dim=1))
        return self.out(torch.cat([gmf, m], dim=1)).squeeze(1)


class VAERec4L(nn.Module):
    def __init__(self, sparse_fields, dense_dim, num_buckets, emb_dim=32, encoder_dims=None, latent_dim=64, dropout=0.1):
        super().__init__()
        if encoder_dims is None:
            encoder_dims = [512, 256, 128, 64]
        self.enc_sparse = SparseEncoder(sparse_fields, num_buckets, int(emb_dim))
        in_dim = int(dense_dim) + int(emb_dim) * len(sparse_fields)
        self.encoder, enc_out_dim = _build_mlp_stack(in_dim, encoder_dims, dropout=dropout)
        self.mu = nn.Linear(enc_out_dim, int(latent_dim))
        self.logvar = nn.Linear(enc_out_dim, int(latent_dim))

        dec_dims = list(reversed([int(d) for d in encoder_dims]))
        self.decoder, dec_out_dim = _build_mlp_stack(int(latent_dim), dec_dims, dropout=dropout)
        self.out = nn.Linear(dec_out_dim, 1)

    def forward(self, dense, sparse):
        x = torch.cat([dense, self.enc_sparse(sparse)], dim=1)
        h = self.encoder(x)
        mu = self.mu(h)
        logvar = self.logvar(h)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        h_dec = self.decoder(z)
        logit = self.out(h_dec).squeeze(1)
        return logit, mu, logvar


def build_model_from_cfg(name: str, sparse_fields: List[str], dense_dim: int, num_buckets: int, cfg: dict):
    n = name.upper()
    dropout = float(cfg.get("dropout", 0.1))
    if n == "MLP":
        return MLPRec4L(
            sparse_fields=sparse_fields,
            dense_dim=dense_dim,
            num_buckets=num_buckets,
            emb_dim=int(cfg.get("emb_dim", 32)),
            hidden_dims=cfg.get("hidden_dims", [512, 256, 128, 64]),
            dropout=dropout,
        )
    if n == "NCF":
        if "user_id" not in sparse_fields or "item_id" not in sparse_fields:
            raise ValueError("NCF requires user_id and item_id")
        return NCFRec4L(
            num_buckets=num_buckets,
            emb_dim=int(cfg.get("emb_dim", 64)),
            mlp_hidden_dims=cfg.get("mlp_hidden_dims", [256, 128, 64, 32]),
            dropout=dropout,
        )
    if n == "VAE":
        return VAERec4L(
            sparse_fields=sparse_fields,
            dense_dim=dense_dim,
            num_buckets=num_buckets,
            emb_dim=int(cfg.get("emb_dim", 32)),
            encoder_dims=cfg.get("encoder_dims", [512, 256, 128, 64]),
            latent_dim=int(cfg.get("latent_dim", 64)),
            dropout=dropout,
        )
    raise ValueError(f"Unknown model: {name}")


def _subset_prepared(p: PreparedData, idx: torch.Tensor) -> PreparedData:
    return PreparedData(
        dense=p.dense[idx],
        sparse={k: v[idx] for k, v in p.sparse.items()},
        labels=p.labels[idx],
        sample_ids=p.sample_ids[idx],
        sparse_fields=p.sparse_fields,
        dense_fields=p.dense_fields,
    )


def make_user_disjoint_loaders(prepared: PreparedData, batch_size: int = 256, val_ratio: float = 0.2, seed: int = 42):
    if "user_id" not in prepared.sparse:
        tr, tr_eval, va = make_loaders(prepared, batch_size=batch_size, val_ratio=val_ratio, seed=seed)
        return tr, tr_eval, va, {
            "split_mode": "sample_random_fallback",
            "train_users": -1,
            "val_users": -1,
            "user_overlap": -1,
        }

    uid = prepared.sparse["user_id"].detach().cpu().numpy().astype(np.int64)
    uniq_users = np.unique(uid)

    rng = np.random.RandomState(seed)
    rng.shuffle(uniq_users)
    n_val_users = max(1, int(len(uniq_users) * float(val_ratio)))

    val_users = set(uniq_users[:n_val_users].tolist())
    train_users = set(uniq_users[n_val_users:].tolist())

    tr_mask = np.isin(uid, list(train_users))
    va_mask = np.isin(uid, list(val_users))

    if tr_mask.sum() == 0 or va_mask.sum() == 0:
        tr, tr_eval, va = make_loaders(prepared, batch_size=batch_size, val_ratio=val_ratio, seed=seed)
        return tr, tr_eval, va, {
            "split_mode": "sample_random_fallback_empty_side",
            "train_users": int(len(train_users)),
            "val_users": int(len(val_users)),
            "user_overlap": int(len(train_users & val_users)),
        }

    tr_idx = torch.from_numpy(np.where(tr_mask)[0]).long()
    va_idx = torch.from_numpy(np.where(va_mask)[0]).long()

    train_d = _subset_prepared(prepared, tr_idx)
    val_d = _subset_prepared(prepared, va_idx)

    train_loader = DataLoader(RecDataset(train_d), batch_size=batch_size, shuffle=True)
    train_eval_loader = DataLoader(RecDataset(train_d), batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(RecDataset(val_d), batch_size=batch_size, shuffle=False)

    return train_loader, train_eval_loader, val_loader, {
        "split_mode": "user_disjoint",
        "train_users": int(len(train_users)),
        "val_users": int(len(val_users)),
        "user_overlap": int(len(train_users & val_users)),
    }


all_summary = []
all_checkpoints = {}

for model_name in MODELS:
    cfg = resolve_hparams(model_name)

    train_loader, train_eval_loader, val_loader, split_info = make_user_disjoint_loaders(
        prepared,
        batch_size=int(cfg["batch_size"]),
        val_ratio=float(cfg["val_ratio"]),
        seed=int(cfg["seed"]),
    )
    train_labels = train_loader.dataset.prepared.labels.float()
    val_labels = val_loader.dataset.prepared.labels.float()
    train_samples = int(train_labels.shape[0])
    val_samples = int(val_labels.shape[0])
    train_pos_rate = float(train_labels.mean().item()) if train_samples > 0 else 0.0
    val_pos_rate = float(val_labels.mean().item()) if val_samples > 0 else 0.0
    print(f"{model_name} data: train_samples={train_samples}, val_samples={val_samples}, train_pos_rate={train_pos_rate:.6f}, val_pos_rate={val_pos_rate:.6f}")

    model = build_model_from_cfg(
        name=model_name,
        sparse_fields=prepared.sparse_fields,
        dense_dim=prepared.dense.shape[1],
        num_buckets=NUM_BUCKETS,
        cfg=cfg,
    )

    model_ckpt_dir = STEP2_DIR / f"{model_name.lower()}_checkpoints"
    history, checkpoints, ckpt_paths = train_model(
        model,
        train_loader,
        val_loader,
        epochs=int(cfg["epochs"]),
        lr=float(cfg["lr"]),
        device=DEVICE,
        early_stop_patience=int(cfg["early_stop_patience"]),
        early_stop_min_delta=float(cfg["early_stop_min_delta"]),
        save_end_only=bool(cfg["checkpoint_end_only"]),
        checkpoint_interval_epochs=float(cfg["checkpoint_interval_epochs"]),
        use_class_balance=bool(cfg["use_class_balance"]),
        weight_decay=float(cfg["weight_decay"]),
        max_grad_norm=float(cfg["max_grad_norm"]),
        lr_scheduler_patience=int(cfg["lr_scheduler_patience"]),
        lr_scheduler_factor=float(cfg["lr_scheduler_factor"]),
        min_lr=float(cfg["min_lr"]),
        checkpoint_dir=model_ckpt_dir,
        checkpoint_prefix=model_name.lower(),
    )
    val = evaluate_model(model, val_loader, device=DEVICE)

    save_table(history, str(STEP2_DIR / f"{model_name.lower()}_train_history.csv"))

    final_state_path = STEP2_DIR / f"{model_name.lower()}_final_state.pt"
    torch.save(model.state_dict(), final_state_path)
    all_checkpoints[model_name] = ckpt_paths

    best_auc_hist = float(history["auc"].max()) if len(history) > 0 and "auc" in history.columns else float("nan")
    best_epoch_hist = int(history.loc[history["auc"].idxmax(), "epoch"]) if len(history) > 0 and "auc" in history.columns else -1

    all_summary.append({
        "run_timestamp": RUN_TIMESTAMP,
        "model": model_name,
        "epochs": int(cfg["epochs"]),
        "lr": float(cfg["lr"]),
        "batch_size": int(cfg["batch_size"]),
        "val_ratio": float(cfg["val_ratio"]),
        "seed": int(cfg["seed"]),
        "early_stop_patience": int(cfg["early_stop_patience"]),
        "early_stop_min_delta": float(cfg["early_stop_min_delta"]),
        "checkpoint_end_only": bool(cfg["checkpoint_end_only"]),
        "checkpoint_interval_epochs": float(cfg["checkpoint_interval_epochs"]),
        "use_class_balance": bool(cfg["use_class_balance"]),
        "weight_decay": float(cfg["weight_decay"]),
        "max_grad_norm": float(cfg["max_grad_norm"]),
        "lr_scheduler_patience": int(cfg["lr_scheduler_patience"]),
        "lr_scheduler_factor": float(cfg["lr_scheduler_factor"]),
        "min_lr": float(cfg["min_lr"]),
        "dropout": float(cfg.get("dropout", 0.1)),
        "emb_dim": int(cfg.get("emb_dim", -1)),
        "hidden_dims": str(cfg.get("hidden_dims", cfg.get("mlp_hidden_dims", cfg.get("encoder_dims", [])))),
        "latent_dim": int(cfg.get("latent_dim", -1)),
        "num_checkpoints": int(len(ckpt_paths)),
        "split_mode": split_info["split_mode"],
        "split_train_users": split_info["train_users"],
        "split_val_users": split_info["val_users"],
        "split_user_overlap": split_info["user_overlap"],
        "train_samples": train_samples,
        "val_samples": val_samples,
        "train_pos_rate": train_pos_rate,
        "val_pos_rate": val_pos_rate,
        "best_hist_auc": best_auc_hist,
        "best_hist_epoch": best_epoch_hist,
        "val_loss": val["loss"],
        "val_acc": val["acc"],
        "val_auc": val["auc"],
    })

    print(
        model_name,
        "cfg=", cfg,
        "split=", split_info,
        "best_hist_auc=", best_auc_hist,
        "val=", val,
        "num_checkpoints=", len(ckpt_paths),
    )

save_table(all_summary, str(STEP2_DIR / "summary.csv"))
torch.save(all_checkpoints, STEP2_DIR / "all_checkpoints.pt")

latest_file = STEP2_ROOT / "latest_run.txt"
latest_file.write_text(RUN_TIMESTAMP, encoding="utf-8")

print("Saved training artifacts to:", STEP2_DIR)
print("Updated latest pointer:", latest_file)
pd.DataFrame(all_summary)

MLP data: train_samples=622064, val_samples=155754, train_pos_rate=0.500000, val_pos_rate=0.500000
new new


c:\Users\vanvan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


{'epoch': 1, 'train_loss': 0.6901401969434094, 'lr': 0.0002, 'loss': 0.6818254901270561, 'acc': 0.5607368029071484, 'auc': 0.5845869052139969}
{'epoch': 2, 'train_loss': 0.6711550738886982, 'lr': 0.0002, 'loss': 0.6614648858314665, 'acc': 0.5996956739473785, 'auc': 0.6401911532538919}
{'epoch': 3, 'train_loss': 0.6494754429274882, 'lr': 0.0002, 'loss': 0.6468098755736265, 'acc': 0.6236950575908163, 'auc': 0.673133822311129}
{'epoch': 4, 'train_loss': 0.6321617342868384, 'lr': 0.0002, 'loss': 0.6352587292347048, 'acc': 0.6371906981522144, 'auc': 0.6904942188666298}
{'epoch': 5, 'train_loss': 0.6194482308383489, 'lr': 0.0002, 'loss': 0.6294328187682554, 'acc': 0.6433285822514991, 'auc': 0.7001159734127683}
{'epoch': 6, 'train_loss': 0.6097211534831807, 'lr': 0.0002, 'loss': 0.6265543784903383, 'acc': 0.646821269437703, 'auc': 0.7035744276364188}
{'epoch': 7, 'train_loss': 0.6019462090362268, 'lr': 0.0002, 'loss': 0.6262764419338778, 'acc': 0.6489785174056525, 'auc': 0.7057306178660578}
{

c:\Users\vanvan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


{'epoch': 1, 'train_loss': 0.69156351517575, 'lr': 0.0002, 'loss': 0.687411772304372, 'acc': 0.5492635823157029, 'auc': 0.5668805906371944}
{'epoch': 2, 'train_loss': 0.6787909136506773, 'lr': 0.0002, 'loss': 0.6724609913692882, 'acc': 0.584049205798888, 'auc': 0.6166160031368637}
{'epoch': 3, 'train_loss': 0.6618669056644649, 'lr': 0.0002, 'loss': 0.6571898580771949, 'acc': 0.6097435699885717, 'auc': 0.6513298967931721}
{'epoch': 4, 'train_loss': 0.6451401581823459, 'lr': 0.0002, 'loss': 0.6448680032258747, 'acc': 0.6275344453432977, 'auc': 0.6761788635970863}
{'epoch': 5, 'train_loss': 0.6307012832223942, 'lr': 0.0002, 'loss': 0.6355705326805365, 'acc': 0.6393030034541649, 'auc': 0.6912334963274104}
{'epoch': 6, 'train_loss': 0.61881370594596, 'lr': 0.0002, 'loss': 0.6297914456264139, 'acc': 0.6458646326900112, 'auc': 0.6991485295145006}
{'epoch': 7, 'train_loss': 0.609735093405838, 'lr': 0.0002, 'loss': 0.6260880210814609, 'acc': 0.6490234600716515, 'auc': 0.7044806069690037}
{'epoc

c:\Users\vanvan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


{'epoch': 1, 'train_loss': 0.6921917969480385, 'lr': 0.0002, 'loss': 0.6785463642799991, 'acc': 0.5702903296223532, 'auc': 0.597359611731812}
{'epoch': 2, 'train_loss': 0.6646968184400756, 'lr': 0.0002, 'loss': 0.6568829059013592, 'acc': 0.607778933446332, 'auc': 0.6511995123678322}
{'epoch': 3, 'train_loss': 0.6418605855304845, 'lr': 0.0002, 'loss': 0.6429532725235512, 'acc': 0.6270272352555953, 'auc': 0.6782738974257084}
{'epoch': 4, 'train_loss': 0.6248619625982874, 'lr': 0.0002, 'loss': 0.6344929651673792, 'acc': 0.6376401248122039, 'auc': 0.6923912455675206}
{'epoch': 5, 'train_loss': 0.6118704322363823, 'lr': 0.0002, 'loss': 0.6303166681028939, 'acc': 0.6423719455038073, 'auc': 0.6979019337189092}
{'epoch': 6, 'train_loss': 0.6011042393999442, 'lr': 0.0002, 'loss': 0.6307258921597392, 'acc': 0.6435982382474928, 'auc': 0.6997154597710216}
{'epoch': 7, 'train_loss': 0.5905731687707302, 'lr': 0.0002, 'loss': 0.6320127879555394, 'acc': 0.6442659578566201, 'auc': 0.6997742355600785}
{

,run_timestamp,model,epochs,lr,batch_size,val_ratio,seed,early_stop_patience,early_stop_min_delta,checkpoint_end_only,...,split_user_overlap,train_samples,val_samples,train_pos_rate,val_pos_rate,best_hist_auc,best_hist_epoch,val_loss,val_acc,val_auc
0,20260327_015343,MLP,60,0.0002,256,0.2,91,4,0.00005,False,...,0,622064,155754,0.5,0.5,0.706815,8,0.624672,0.649190,0.706815
1,20260327_015343,VAE,60,0.0002,256,0.2,91,4,0.00005,False,...,0,622064,155754,0.5,0.5,0.706132,8,0.625163,0.649306,0.706169
2,20260327_015343,NCF,50,0.0002,256,0.2,91,4,0.00005,False,...,0,622064,155754,0.5,0.5,0.699774,7,0.632013,0.644266,0.699774


In [5]:
import inspect
print("cwd:", Path.cwd())
print("train_model signature:", inspect.signature(train_model))
print("train_model file:", inspect.getsourcefile(train_model))

cwd: h:\TZB\recacc\notebooks
train_model signature: (model: torch.nn.modules.module.Module, train_loader: torch.utils.data.dataloader.DataLoader, val_loader: torch.utils.data.dataloader.DataLoader, epochs: int = 3, lr: float = 0.001, device: str = 'cuda', verbose: bool = True, early_stop_patience: int = 0, early_stop_min_delta: float = 0.0, save_end_only: bool = False, checkpoint_interval_epochs: float = 0.5, use_class_balance: bool = True, weight_decay: float = 1e-05, max_grad_norm: float = 5.0, lr_scheduler_patience: int = 2, lr_scheduler_factor: float = 0.5, min_lr: float = 1e-06, checkpoint_dir=None, checkpoint_prefix: str = 'model')
train_model file: C:\Users\vanvan\AppData\Local\Temp\ipykernel_33788\4202246870.py
